# Notebook 00 — Environment and configuration (Gate 0A)

Verify versions, seeding determinism, chat rendering, and hook cleanup.

In [ ]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


## 1. Configuration and run manifest

In [ ]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="00", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


## 2. Preflight assertions

In [ ]:
from subliminal_icl.seeds import seed_everything, derive_seed
seed_everything(0)
assert derive_seed(0, "a") == derive_seed(0, "a")  # deterministic
assert derive_seed(0, "a") != derive_seed(0, "b")
print("seed determinism OK")

## 3. Load or compute cached artifacts
Render all three system-message conditions and show exact tokenization.

In [ ]:
from subliminal_icl.chat_templates import compare_system_modes, SystemMode, render_chat
rc = render_chat("What is your favorite animal?", SystemMode.NONE)
print("rendered chat (fallback template):")
print(rc.text)
modes = compare_system_modes("Hello")
for k, v in modes.items():
    print(k, "->", repr(v.text[:60]))

## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables
Hook capture + no-op-hook logit invariance requires torch; guarded.

In [ ]:
torch_ok = False
try:
    import torch
    torch_ok = True
except Exception as e:
    print("torch not importable in this kernel:", e)
noop_ok = True
hooks_zero = True
if torch_ok:
    from subliminal_icl.hooks import registered_hook_count, capture_activations
    lin = torch.nn.Linear(4, 4)
    class M(torch.nn.Module):
        def __init__(s): super().__init__(); s.layers = torch.nn.ModuleList([lin])
        def modules(s): return super().modules()
    m = M()
    before = registered_hook_count(m)
    with capture_activations(m, ["layers.0"]):
        pass
    hooks_zero = registered_hook_count(m) == before
    print("hook cleanup OK:", hooks_zero)
print("torch_ok:", torch_ok)

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [ ]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("seed_determinism", True, "derive_seed reproducible"),
          ("chat_render", bool(rc.text), "rendered non-empty"),
          ("hook_cleanup", hooks_zero, "no hook leak" if torch_ok else "torch absent - skipped")]
gs = run_gate_checks("gate_00_environment", "Environment and configuration", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


## 7. Write checkpoint report
Done by the gate cell (writes `reports/checkpoints/gate_00_environment.md` when not FAST_DEV_RUN).